# بهبود کد کلاسی برای پروژه تشخیص صدا

در نوت‌بوک کلاس، عامل جاروبرقی در محیط چهار اتاق شبیه‌سازی شده بود. در این نسخه، ساختار آزمایش و ارزیابی حفظ شده، اما مسئله به تشخیص صدا تغییر کرده است: فایل‌های صوتی خوانده می‌شوند، از آن‌ها ویژگی استخراج می‌شود، یک مدل پایه آموزش می‌بیند و نتیجه با معیارهای ارزیابی بررسی می‌شود.

**نکته:** برای اجرای آموزش، باید مجموعه‌داده صوتی خود را در پوشه `data/train/` قرار دهید؛ هر کلاس صدا یک زیرپوشه جداگانه دارد. بدون داده واقعی، مدل آموزش داده نمی‌شود و نباید دقتی به‌عنوان نتیجه واقعی گزارش کرد.

## ۱. نصب و وارد کردن کتابخانه‌ها

در صورت نیاز، در ترمینال محیط پایتون اجرا کنید: `python -m pip install numpy matplotlib librosa scikit-learn soundfile`

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path('data/train')
TEST_SIZE = 0.25
RANDOM_STATE = 42

print('Python:', sys.version.split()[0])
print('Executable:', sys.executable)
print('Dataset directory:', DATA_DIR.resolve())

## ۲. بررسی ساختار مجموعه‌داده

ساختار پیشنهادی:

```text
data/train/
├── dog_bark/
│   ├── sample1.wav
│   └── sample2.wav
├── siren/
│   ├── sample1.wav
│   └── sample2.wav
└── speech/
    ├── sample1.wav
    └── sample2.wav
```

نام پوشه‌ها به‌عنوان برچسب کلاس استفاده می‌شود. بهتر است هر کلاس چندین نمونه متنوع داشته باشد.

In [ ]:
audio_extensions = {'.wav', '.flac', '.ogg'}
if not DATA_DIR.exists():
    print('پوشه داده پیدا نشد:', DATA_DIR.resolve())
else:
    class_dirs = sorted(p for p in DATA_DIR.iterdir() if p.is_dir())
    if not class_dirs:
        print('هنوز پوشه‌ای برای کلاس‌های صدا وجود ندارد. پوشه‌هایی مانند speech و siren بسازید.')
    else:
        for class_dir in class_dirs:
            files = [p for p in class_dir.rglob('*') if p.suffix.lower() in audio_extensions]
            print(f'{class_dir.name}: {len(files)} فایل صوتی')

## ۳. استخراج ویژگی‌های صوتی

برای یک مدل پایه، از ویژگی‌های MFCC استفاده می‌کنیم. MFCC نمایشی فشرده از ویژگی‌های طیفی صداست. برای هر فایل، میانگین و انحراف معیار ضرایب MFCC محاسبه می‌شود تا همه فایل‌ها به بردارهای با طول ثابت تبدیل شوند.

In [ ]:
try:
    import librosa
except ImportError as exc:
    raise ImportError('کتابخانه librosa نصب نیست. دستور نصب را در بخش اول اجرا کنید.') from exc

def extract_features(file_path, sample_rate=22050, n_mfcc=20):
    """استخراج ویژگی MFCC از یک فایل صوتی؛ در صورت خطا، خطا به فراخواننده منتقل می‌شود."""
    signal, sr = librosa.load(file_path, sr=sample_rate, mono=True)
    if signal.size == 0:
        raise ValueError(f'فایل صوتی خالی است: {file_path}')
    mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=n_mfcc)
    features = np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)])
    return features

## ۴. ساخت داده‌های ویژگی و برچسب‌ها

فایل‌های خراب یا غیرقابل‌خواندن گزارش می‌شوند و از آموزش کنار گذاشته می‌شوند. اگر داده کافی وجود نداشته باشد، اجرای آموزش متوقف می‌شود تا نتیجه گمراه‌کننده تولید نشود.

In [ ]:
X, y, skipped_files = [], [], []
if DATA_DIR.exists():
    for class_dir in sorted(p for p in DATA_DIR.iterdir() if p.is_dir()):
        for audio_path in sorted(class_dir.rglob('*')):
            if audio_path.suffix.lower() not in audio_extensions:
                continue
            try:
                X.append(extract_features(audio_path))
                y.append(class_dir.name)
            except Exception as exc:
                skipped_files.append((str(audio_path), str(exc)))

print('تعداد نمونه‌های قابل استفاده:', len(X))
print('تعداد فایل‌های ردشده:', len(skipped_files))
for file_name, error in skipped_files[:5]:
    print('-', file_name, '|', error)

if len(X) > 0:
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y)
    print('شکل ماتریس ویژگی‌ها:', X.shape)
    print('کلاس‌ها:', sorted(set(y)))
else:
    print('برای آموزش، فایل‌های صوتی را در پوشه‌های کلاس قرار دهید و سلول‌ها را دوباره اجرا کنید.')

## ۵. آموزش و ارزیابی مدل

از Random Forest به‌عنوان مدل پایه استفاده می‌کنیم. داده آزمون جدا نگه داشته می‌شود. اگر هر کلاس نمونه کافی نداشته باشد، تقسیم طبقه‌بندی‌شده ممکن است ممکن نباشد؛ در این حالت باید داده بیشتری جمع‌آوری شود.

In [ ]:
if len(X) > 0:
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

    unique_classes, counts = np.unique(y, return_counts=True)
    if len(unique_classes) < 2:
        print('برای طبقه‌بندی، حداقل دو کلاس صوتی لازم است.')
    elif counts.min() < 2:
        print('هر کلاس باید دست‌کم دو نمونه داشته باشد تا ارزیابی جداگانه انجام شود.')
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
        )
        model = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                       class_weight='balanced')
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        print(f'Accuracy: {accuracy_score(y_test, y_pred):.3f}')
        print('\nگزارش طبقه‌بندی:')
        print(classification_report(y_test, y_pred, zero_division=0))
        ConfusionMatrixDisplay.from_predictions(y_test, y_pred, xticks_rotation=45)
        plt.title('Confusion Matrix — Sound Classification')
        plt.tight_layout()
        plt.show()
else:
    print('آموزش انجام نشد، چون داده صوتی در دسترس نیست.')

## ۶. آزمایش یک فایل صوتی جدید

پس از اجرای موفق بخش آموزش، مسیر فایل را در متغیر `new_audio_path` قرار دهید. فایل باید با همان روش پیش‌پردازش شود.

In [ ]:
new_audio_path = Path('data/new_sample.wav')  # این مسیر را با مسیر فایل خود عوض کنید

if 'model' in globals() and new_audio_path.exists():
    new_features = extract_features(new_audio_path).reshape(1, -1)
    predicted_class = model.predict(new_features)[0]
    print('کلاس تشخیص‌داده‌شده:', predicted_class)
    if hasattr(model, 'predict_proba'):
        confidence = float(model.predict_proba(new_features).max())
        print(f'امتیاز اطمینان مدل: {confidence:.3f}')
else:
    print('ابتدا مدل را با داده کافی آموزش دهید و مسیر فایل صوتی جدید را بررسی کنید.')

## ۷. تغییرات اعمال‌شده نسبت به کد کلاس

1. شبیه‌سازی اتاق‌های کثیف و عامل‌های `random/reflex/model` با مسئله واقعی‌تر دریافت و طبقه‌بندی فایل صوتی جایگزین شد.
2. به‌جای امتیاز پاک‌بودن اتاق‌ها، معیارهای Accuracy، Precision، Recall، F1-score و ماتریس درهم‌ریختگی استفاده شدند.
3. استخراج ویژگی MFCC اضافه شد تا صدا به داده عددی قابل استفاده برای مدل تبدیل شود.
4. نام پوشه‌ها به‌عنوان برچسب استفاده می‌شود؛ بنابراین افزودن کلاس‌های جدید بدون تغییر منطق اصلی امکان‌پذیر است.
5. مدیریت فایل‌های خراب، ورودی خالی، نبود داده و کمبود نمونه برای ارزیابی اضافه شد.
6. مسیر پیش‌بینی یک فایل جدید پس از آموزش مدل اضافه شد.

**محدودیت:** این یک خط پایه آموزشی است. کیفیت نهایی به حجم و تنوع داده‌ها، تعریف کلاس‌ها و شرایط ضبط بستگی دارد. نتیجه آزمایش باید فقط پس از اجرای نوت‌بوک روی داده واقعی گزارش شود.